# 02b — Generator Cost Parameters

**Purpose:** Attach heat rates, fuel costs, and O&M costs to every generator snapped to a bus.

**Outputs:**
- `data/processed/generators_with_costs.parquet` — one row per generator with all cost fields
- `data/processed/cost_coverage.csv` — one row per technology with coverage statistics

**Data sources:**
- Heat rates: EIA `electricity/facility-fuel` (Form EIA-923, annual 2024)
- Fuel costs: EIA `electricity/electric-power-operational-data` (annual 2024, state×fuel level, consumption-weighted avg)
- O&M: NREL ATB 2024 hardcoded table

**Hard limits:**
- Never paginate beyond offset 30,000 on any single EIA endpoint
- If any single operation exceeds 50 iterations, stop and aggregate what is available
- ATB values are hardcoded — no spreadsheet fetch

In [ ]:
import sys, json
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import utils

BG        = PROJECT_ROOT / "data" / "processed" / "background files"
PROCESSED = PROJECT_ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

PAGE_SIZE  = 5_000
MAX_OFFSET = 30_000

print("Imports OK")

## Step 0 — Load and verify required files

In [ ]:
print("─" * 60)
print("STEP 0 — Load files")
print("─" * 60)

plants = gpd.read_file(BG / "power_plants.geojson")
buses  = gpd.read_file(BG / "bus_locations.geojson")

print(f"  power_plants : {plants.shape}")
print(f"  bus_locations: {buses.shape}")
print(f"  Heat-rate columns in power_plants: {[c for c in plants.columns if 'heat' in c.lower()]}")

# Report if generators_with_costs.parquet already exists
genwc_path = PROCESSED / "generators_with_costs.parquet"
if genwc_path.exists():
    existing = pd.read_parquet(genwc_path)
    print(f"\n  generators_with_costs.parquet already exists: {existing.shape}")
    print(f"  Columns: {existing.columns.tolist()}")
    print(existing.head(3).to_string())

## Step 1 — Load giant component from grid network

In [ ]:
print("─" * 60)
print("STEP 1 — Load giant component")
print("─" * 60)

G = nx.read_graphml(BG / "grid_network.graphml")
comps = sorted(nx.connected_components(G), key=len, reverse=True)
giant_nodes = {int(n) for n in comps[0]}
print(f"  Giant component: {len(giant_nodes):,} buses  ({len(comps):,} total components)")

## Step 2 — Snap plants to buses

In [ ]:
print("─" * 60)
print("STEP 2 — Snap plants to buses (≤50 km)")
print("─" * 60)

MAX_SNAP_M = 50_000

plants_proj = plants.to_crs(epsg=5070)
buses_proj  = buses[['bus_id', 'geometry']].to_crs(epsg=5070)

snapped = gpd.sjoin_nearest(
    plants_proj,
    buses_proj,
    how='left',
    max_distance=MAX_SNAP_M,
    distance_col='snap_dist_m',
)
snapped = (
    snapped
    .sort_values('snap_dist_m', na_position='last')
    .drop_duplicates(subset=['plantid', 'generatorid', 'period'], keep='first')
)

excluded = snapped[snapped['bus_id'].isna()]
included = snapped[snapped['bus_id'].notna()].copy()
included['bus_id'] = included['bus_id'].astype(int)
included['in_giant_component'] = included['bus_id'].isin(giant_nodes)
included['capacity_mw'] = pd.to_numeric(included['nameplate-capacity-mw'], errors='coerce')

print(f"  Snapped  : {len(included):,}")
print(f"  Excluded : {len(excluded):,}  (> {MAX_SNAP_M/1000:.0f} km or no nearby bus)")
print(f"  In-giant : {included['in_giant_component'].sum():,}  "
      f"({100*included['in_giant_component'].mean():.1f}%)")

## Step 3 — Heat Rates from EIA facility-fuel

`electricity/facility-fuel` (Form EIA-923) provides plant+prime-mover level annual fuel
consumption (MMBtu) and net generation (MWh). Heat rate = consumption_mmbtu / generation_mwh.

No heat-rate column exists in `power_plants.geojson`, so we fetch from the API.
Paginates until complete or offset 30,000. Aggregates across prime movers by plant+fuel.

In [ ]:
print("─" * 60)
print("STEP 3 — Heat rates (facility-fuel, annual 2024)")
print("─" * 60)

NO_HEAT_RATE_FUELS = {
    'WAT', 'WND', 'SUN', 'GEO', 'ES', 'PH', 'FW', 'WH', 'PUR', 'MWH',
}

hr_records = []
offset = 0
iteration = 0
while True:
    iteration += 1
    if iteration > 50:
        print(f"  ⚠ Reached 50-iteration limit at offset {offset}, stopping.")
        break
    if offset > MAX_OFFSET:
        print(f"  ⚠ Reached max offset {MAX_OFFSET}, stopping.")
        break
    try:
        resp = utils.eia_get('electricity/facility-fuel/data', {
            'frequency'           : 'annual',
            'start'               : '2024',
            'end'                 : '2024',
            'data[0]'             : 'generation',
            'data[1]'             : 'consumption-for-eg-btu',
            'sort[0][column]'     : 'plantCode',
            'sort[0][direction]'  : 'asc',
            'sort[1][column]'     : 'fuel2002',
            'sort[1][direction]'  : 'asc',
            'length'              : PAGE_SIZE,
            'offset'              : offset,
        })
        records = resp.get('response', {}).get('data', [])
    except Exception as e:
        print(f"  Error at offset {offset}: {e}  — stopping.")
        break
    hr_records.extend(records)
    print(f"  Offset {offset:>6,}: fetched {len(records):,}  (total: {len(hr_records):,})")
    if len(records) < PAGE_SIZE:
        break
    offset += PAGE_SIZE

ff_df = pd.DataFrame(hr_records)
print(f"  Total facility-fuel rows: {len(ff_df):,}")

In [ ]:
# ── Compute heat rates ──────────────────────────────────────────────────────
# Units: consumption-for-eg-btu is in MMBtu (confirmed from API units field)
# heat_rate = MMBtu / MWh directly

ff_df['generation_mwh']    = pd.to_numeric(ff_df['generation'],             errors='coerce')
ff_df['consumption_mmbtu'] = pd.to_numeric(ff_df['consumption-for-eg-btu'], errors='coerce')

# Remove aggregate 'ALL' rows
ff_fuel = ff_df[
    (ff_df['fuel2002']   != 'ALL') &
    (ff_df['primeMover'] != 'ALL')
].copy()

# Aggregate across prime movers → plant+fuel level
ff_plant = (
    ff_fuel
    .groupby(['plantCode', 'fuel2002'], as_index=False)
    .agg(generation_mwh=('generation_mwh', 'sum'),
         consumption_mmbtu=('consumption_mmbtu', 'sum'))
)

ff_plant['heat_rate_mmbtu_mwh'] = np.where(
    ff_plant['generation_mwh'] >= 1.0,
    ff_plant['consumption_mmbtu'] / ff_plant['generation_mwh'],
    np.nan,
)

# Sanity filter: plausible heat rate range 3–50 MMBtu/MWh
mask_valid = (
    (ff_plant['heat_rate_mmbtu_mwh'] >= 3) &
    (ff_plant['heat_rate_mmbtu_mwh'] <= 50)
)
ff_valid = ff_plant[mask_valid].copy()
ff_valid['plantCode'] = ff_valid['plantCode'].astype(str)
hr_lookup = ff_valid[['plantCode', 'fuel2002', 'heat_rate_mmbtu_mwh']].rename(
    columns={'plantCode': 'plantid_str', 'fuel2002': 'energy_source_code'}
)
print(f"  Plant+fuel rows after prime-mover aggregation: {len(ff_plant):,}")
print(f"  Valid heat-rate lookup entries (3–50 MMBtu/MWh): {len(hr_lookup):,}")

# Join to generators
included['plantid_str'] = included['plantid'].astype(str)
included2 = included.merge(hr_lookup, on=['plantid_str', 'energy_source_code'], how='left')
# Explicitly null for non-thermal fuels
included2.loc[
    included2['energy_source_code'].isin(NO_HEAT_RATE_FUELS), 'heat_rate_mmbtu_mwh'
] = np.nan

n_thermal = (~included2['energy_source_code'].isin(NO_HEAT_RATE_FUELS)).sum()
n_with_hr = included2['heat_rate_mmbtu_mwh'].notna().sum()
print(f"  Thermal gens: {n_thermal:,}  |  With heat rate: {n_with_hr:,}  "
      f"({100*n_with_hr/max(n_thermal,1):.1f}%)")
print(f"\n  Heat rate distribution:")
print(included2['heat_rate_mmbtu_mwh'].describe().to_string())

## Step 4 — Fuel Costs from EIA electric-power-operational-data

`electricity/electric-power-operational-data` provides state×fuel annual cost and consumption.
We compute a consumption-weighted average of `cost-per-btu` per state×fuel type,
then join to generators by `stateid` × mapped fuel type.
Missing state matches fall back to the national consumption-weighted average.

Note: This endpoint is state-level (no plantCode). The weighted aggregation is by
`location`+`fueltypeid` using `consumption-for-eg-btu` as the weight.

In [ ]:
print("─" * 60)
print("STEP 4 — Fuel costs (electric-power-operational-data, annual 2024)")
print("─" * 60)

cost_records = []
offset = 0
iteration = 0
while True:
    iteration += 1
    if iteration > 50:
        print(f"  ⚠ Reached 50-iteration limit at offset {offset}, stopping.")
        break
    if offset > MAX_OFFSET:
        print(f"  ⚠ Reached max offset {MAX_OFFSET}, stopping.")
        break
    try:
        resp = utils.eia_get('electricity/electric-power-operational-data/data', {
            'frequency'           : 'annual',
            'start'               : '2024',
            'end'                 : '2024',
            'data[0]'             : 'cost-per-btu',
            'data[1]'             : 'consumption-for-eg-btu',
            'sort[0][column]'     : 'location',
            'sort[0][direction]'  : 'asc',
            'length'              : PAGE_SIZE,
            'offset'              : offset,
        })
        records = resp.get('response', {}).get('data', [])
    except Exception as e:
        print(f"  Error at offset {offset}: {e}  — stopping.")
        break
    if not records:
        print(f"  Empty response at offset {offset}, stopping.")
        break
    cost_records.extend(records)
    print(f"  Offset {offset:>6,}: fetched {len(records):,}  (total: {len(cost_records):,})")
    if len(records) < PAGE_SIZE:
        break
    offset += PAGE_SIZE

print(f"  Total fuel-cost rows: {len(cost_records):,}")

cost_df_raw = pd.DataFrame(cost_records)
if len(cost_df_raw):
    print(f"  Columns: {cost_df_raw.columns.tolist()}")
    print(cost_df_raw.head(3).to_string())

In [ ]:
# ── Aggregate and join fuel costs ──────────────────────────────────────────

if len(cost_df_raw) and 'location' in cost_df_raw.columns:
    cost_df_raw['cost_per_mmbtu'] = pd.to_numeric(cost_df_raw['cost-per-btu'],          errors='coerce')
    cost_df_raw['qty']            = pd.to_numeric(cost_df_raw['consumption-for-eg-btu'], errors='coerce')

    # Keep 2-char state codes and non-aggregate fuel types with valid costs
    AGGREGATE_FUELTYPES = {'ALL', 'AOR', 'FOS', 'COW', 'COG', 'OOG'}
    cost_df = cost_df_raw[
        cost_df_raw['cost_per_mmbtu'].notna() &
        (cost_df_raw['qty'].fillna(0) > 0) &
        (cost_df_raw['location'].str.len() == 2) &
        (~cost_df_raw['fueltypeid'].isin(AGGREGATE_FUELTYPES))
    ].copy()

    def wavg(group):
        """Consumption-weighted average of cost-per-btu."""
        w = group['qty']
        v = group['cost_per_mmbtu']
        return (v * w).sum() / w.sum() if w.sum() > 0 else np.nan

    state_fuel_cost = (
        cost_df
        .groupby(['location', 'fueltypeid'])
        .apply(wavg, include_groups=False)
        .reset_index(name='fuel_cost_per_mmbtu')
        .rename(columns={'location': 'stateid'})
    )
    national_fuel_cost = (
        cost_df
        .groupby('fueltypeid')
        .apply(wavg, include_groups=False)
        .reset_index(name='national_cost')
    )

    print(f"  State×fuel lookup entries: {len(state_fuel_cost):,}")
    print(f"\n  National consumption-weighted fuel costs ($/MMBtu):")
    print(national_fuel_cost.to_string())

    # Map EIA energy_source_code → epod fueltypeid
    ESC_TO_FUELTYPEID = {
        'NG' :'NG',  'OG' :'NG',  'BFG':'NG',  'SGC':'NG',  'PG' :'NG',
        'BIT':'BIT', 'SUB':'SUB', 'LIG':'LIG', 'RC' :'COL', 'ANT':'COL', 'WC':'COL', 'SC':'SUB',
        'DFO':'DFO', 'RFO':'RFO', 'KER':'DFO', 'JF' :'DFO', 'WO' :'RFO', 'OO':'RFO', 'PC':'PC',
        'NUC':'NUC',
        'WDS':'WOO', 'BLQ':'WOO', 'OBS':'OBL', 'AB' :'WAS', 'MSW':'MSW',
        'OBL':'OBL', 'LFG':'LFG', 'OBG':'OBG', 'OG2':'NG',
        'WAT':None,  'WND':None,  'SUN':None,  'GEO':None,
        'ES' :None,  'PH' :None,  'FW' :None,  'WH' :None,  'PUR':None,  'MWH':None,
    }
    included2['_fuel_key'] = included2['energy_source_code'].map(ESC_TO_FUELTYPEID)

    # Primary join: state × fuel type
    included3 = included2.merge(
        state_fuel_cost.rename(columns={'fueltypeid': '_fuel_key'}),
        left_on=['stateid', '_fuel_key'],
        right_on=['stateid', '_fuel_key'],
        how='left',
    )

    # Fill missing with national weighted average
    missing_mask = included3['fuel_cost_per_mmbtu'].isna() & included3['_fuel_key'].notna()
    if missing_mask.sum() > 0:
        nat_map = national_fuel_cost.set_index('fueltypeid')['national_cost'].to_dict()
        included3.loc[missing_mask, 'fuel_cost_per_mmbtu'] = (
            included3.loc[missing_mask, '_fuel_key'].map(nat_map)
        )
        included3['fuel_cost_imputed'] = missing_mask
        print(f"  Imputed {missing_mask.sum():,} generators from national weighted average.")
    else:
        included3['fuel_cost_imputed'] = False

    # No fuel cost for non-thermal fuels
    included3.loc[included3['_fuel_key'].isna(), 'fuel_cost_per_mmbtu'] = np.nan
    fuel_cost_source = 'state'

else:
    print("  ⚠ No fuel cost data fetched; fuel_cost_per_mmbtu will be null.")
    included3 = included2.copy()
    included3['fuel_cost_per_mmbtu'] = np.nan
    included3['fuel_cost_imputed']   = False
    fuel_cost_source = 'none'

print(f"\n  Fuel cost source: {fuel_cost_source}")
n_fc = included3['fuel_cost_per_mmbtu'].notna().sum()
print(f"  Generators with fuel_cost_per_mmbtu: {n_fc:,}")

## Step 5 — O&M Costs (hardcoded ATB 2024 table)

Fixed-O&M (\$/kW-yr) and variable-O&M (\$/MWh) from NREL ATB 2024 Moderate scenario.
Technologies not in the table use the 'All Other' defaults.

In [ ]:
print("─" * 60)
print("STEP 5 — O&M costs (hardcoded ATB table)")
print("─" * 60)

ATB_OM = {
    'Natural Gas Fired Combined Cycle':      {'vom': 3.51,  'fom': 15.13},
    'Natural Gas Fired Combustion Turbine':  {'vom': 4.46,  'fom':  9.89},
    'Conventional Steam Coal':               {'vom': 5.00,  'fom': 35.00},
    'Nuclear':                               {'vom': 2.37,  'fom': 121.00},
    'Conventional Hydroelectric':            {'vom': 2.50,  'fom': 44.00},
    'Onshore Wind Turbine':                  {'vom': 0.00,  'fom': 26.00},
    'Solar Photovoltaic':                    {'vom': 0.00,  'fom': 17.00},
    'Batteries':                             {'vom': 0.00,  'fom': 22.00},
    'All Other':                             {'vom': 5.00,  'fom': 20.00},
}

included3['vom_per_mwh']   = included3['technology'].map(lambda t: ATB_OM.get(t, ATB_OM['All Other'])['vom'])
included3['fom_per_kw_yr'] = included3['technology'].map(lambda t: ATB_OM.get(t, ATB_OM['All Other'])['fom'])

tech_counts = included3['technology'].value_counts().reset_index()
tech_counts.columns = ['technology', 'count']
tech_counts['in_ATB'] = tech_counts['technology'].isin(ATB_OM)
print(tech_counts.to_string())

## Step 6 — Compute Marginal Costs

- Thermal: `marginal_cost = heat_rate × fuel_cost + vom`
- Non-thermal (no heat rate or fuel cost): `marginal_cost = vom`
- Cap at $1,000/MWh

In [ ]:
print("─" * 60)
print("STEP 6 — Compute marginal costs")
print("─" * 60)

df = included3.copy()

df['marginal_cost_per_mwh'] = np.where(
    df['heat_rate_mmbtu_mwh'].notna() & df['fuel_cost_per_mmbtu'].notna(),
    df['heat_rate_mmbtu_mwh'] * df['fuel_cost_per_mmbtu'] + df['vom_per_mwh'],
    df['vom_per_mwh'],   # non-thermal or missing: VOM only
)

hi = df['marginal_cost_per_mwh'] > 1000
lo = df['marginal_cost_per_mwh'] < 0
print(f"  MC > $1000/MWh : {hi.sum():,}  (capped)")
print(f"  MC < $0/MWh    : {lo.sum():,}")
df.loc[hi, 'marginal_cost_per_mwh'] = 1000.0

print(f"\n  Marginal cost distribution:")
print(df['marginal_cost_per_mwh'].describe().to_string())
print()
print("  By technology (mean $/MWh):")
print(
    df.groupby('technology')
    .agg(n=('marginal_cost_per_mwh','count'), mean_mc=('marginal_cost_per_mwh','mean'), total_mw=('capacity_mw','sum'))
    .sort_values('total_mw', ascending=False)
    .to_string()
)

## Step 7 — Build Output DataFrame

In [ ]:
print("─" * 60)
print("STEP 7 — Build output DataFrame")
print("─" * 60)

gen_costs = pd.DataFrame({
    'bus_id'               : df['bus_id'].astype(int),
    'plant_id'             : df['plantid'].astype(str),
    'generator_id'         : df['generatorid'].astype(str),
    'technology'           : df['technology'].astype(str),
    'fuel_type'            : df['energy-source-desc'].astype(str),
    'capacity_mw'          : df['capacity_mw'].astype(float),
    'heat_rate_mmbtu_mwh'  : df['heat_rate_mmbtu_mwh'].astype(float),
    'fuel_cost_per_mmbtu'  : df['fuel_cost_per_mmbtu'].astype(float),
    'vom_per_mwh'          : df['vom_per_mwh'].astype(float),
    'fom_per_kw_yr'        : df['fom_per_kw_yr'].astype(float),
    'marginal_cost_per_mwh': df['marginal_cost_per_mwh'].astype(float),
    'in_giant_component'   : df['in_giant_component'].astype(bool),
})

print(f"  Output shape: {gen_costs.shape}")
print(f"  Columns: {gen_costs.columns.tolist()}")
print()
print(gen_costs.head(3).to_string())

## Step 8 — Save Outputs

In [ ]:
print("─" * 60)
print("STEP 8 — Save outputs")
print("─" * 60)

# ── generators_with_costs.parquet ───────────────────────────────────────────
parquet_path = PROCESSED / "generators_with_costs.parquet"
gen_costs.to_parquet(parquet_path, index=False)
print(f"  Saved generators_with_costs.parquet  ({parquet_path.stat().st_size/1024:.1f} KB)")

# ── cost_coverage.csv ────────────────────────────────────────────────────────
NO_HEAT_RATE_FUELS_ESC = {
    'WAT','WND','SUN','GEO','ES','PH','FW','WH','PUR','MWH',
}
cov_rows = []
for tech, grp in gen_costs.groupby('technology'):
    tech_mask  = df['technology'] == tech
    esc_codes  = df.loc[tech_mask, 'energy_source_code'].unique()

    n_total  = len(grp)
    total_mw = grp['capacity_mw'].sum()

    n_hr_thermal = (~df.loc[tech_mask, 'energy_source_code'].isin(NO_HEAT_RATE_FUELS_ESC)).sum()
    n_with_hr    = grp['heat_rate_mmbtu_mwh'].notna().sum()
    hr_real_frac = (n_with_hr / n_hr_thermal) if n_hr_thermal > 0 else np.nan

    n_with_fc = grp['fuel_cost_per_mmbtu'].notna().sum()
    fc_frac   = (n_with_fc / n_total) if n_total > 0 else 0.0

    mean_mc = grp['marginal_cost_per_mwh'].mean()

    cov_rows.append({
        'technology'                : tech,
        'n_generators'              : n_total,
        'total_mw'                  : round(total_mw, 1),
        'frac_real_heat_rate'       : round(float(hr_real_frac), 4) if pd.notna(hr_real_frac) else np.nan,
        'frac_with_fuel_cost'       : round(float(fc_frac), 4),
        'mean_marginal_cost_per_mwh': round(mean_mc, 2),
    })

coverage_df = pd.DataFrame(cov_rows).sort_values('total_mw', ascending=False)
csv_path    = PROCESSED / "cost_coverage.csv"
coverage_df.to_csv(csv_path, index=False)
print(f"  Saved cost_coverage.csv")

print()
print("  Outputs written:")
print(f"    {parquet_path}")
print(f"    {csv_path}")

## Step 9 — Cost Coverage Summary

In [ ]:
print("─" * 60)
print("STEP 9 — Cost coverage table")
print("─" * 60)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 130)
pd.set_option('display.max_rows', 60)
print(coverage_df.to_string(index=False))